# 0.0 Imports

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.cluster import AffinityPropagation
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import sklearn.metrics as mt

# 0.1 - Load Dataset

In [14]:
actual_folder = Path.cwd()
main_folder = actual_folder.parent.parent
path_dataset = main_folder / 'dataset' / 'clusters_datasets' / 'a_traning' / 'X_dataset.csv'

df = pd.read_csv(path_dataset)
print(f'Dataset carregado: {df.shape[0]} amostras, {df.shape[1]} features')

Dataset carregado: 178 amostras, 13 features


# 0.2 - Pré-processamento

In [15]:
#Dataset para treinamento das variaveis
X = df.values

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [16]:
ap_def = AffinityPropagation(random_state=42)
labels_def = ap_def.fit_predict(X)
n_clusters_def = len(ap_def.cluster_centers_indices_)

print(f'Clusters encontrados (default): {n_clusters_def}')

Clusters encontrados (default): 17


## Passo 2 — Performance no treino (default)

In [17]:
if 1 < n_clusters_def < len(X):
    sil_def = mt.silhouette_score(X, labels_def)
    print(f'Clusters: {n_clusters_def}')
    print(f'Silhouette Score: {sil_def:.4f}')
else:
    sil_def = float('nan')
    print(f'Clusters: {n_clusters_def} — Silhouette Score não aplicável')

Clusters: 17
Silhouette Score: 0.1694


## Passo 3 — Performance na validação (default)

In [18]:
print('Clusterização não possui conjunto de validação separado.')
print('Avaliação feita sobre o dataset completo.')

Clusterização não possui conjunto de validação separado.
Avaliação feita sobre o dataset completo.


## Passo 4 — Ajuste de hiperparâmetros

In [19]:
print(f'{"Preferência":>12}  {"Clusters":>9}  {"Silhouette Score":>16}')
print('-' * 42)

preferences = [-50, -100, -200, -300, -500, -700, -1000]

best_ss, best_pref = -1, preferences[0]
for pref in preferences:
    ap = AffinityPropagation(preference=pref, max_iter=300, random_state=42)
    lbl = ap.fit_predict(X)
    n = len(ap.cluster_centers_indices_)
    if 1 < n < len(X):
        ss = mt.silhouette_score(X, lbl)
        marker = ' <---' if ss > best_ss else ''
        if ss > best_ss:
            best_ss, best_pref = ss, pref
        print(f'{pref:>12}  {n:>9}  {ss:>16.4f}{marker}')
    else:
        print(f'{pref:>12}  {n:>9}  {"N/A":>16}')

print()
print(f'Melhor preference: {best_pref}  |  Silhouette Score: {best_ss:.4f}')

 Preferência   Clusters  Silhouette Score
------------------------------------------
         -50          7            0.2023 <---
        -100          4            0.1588
        -200          3            0.1957
        -300          2            0.1762
        -500          1               N/A
        -700          1               N/A
       -1000          1               N/A

Melhor preference: -50  |  Silhouette Score: 0.2023


## Passo 5 — União treino + validação

In [20]:
print(f'Usando todos os dados para o retreino final: {X.shape}')

Usando todos os dados para o retreino final: (178, 13)


## Passo 6 — Retreino com melhores parâmetros

In [21]:
ap_final = AffinityPropagation(preference=best_pref, max_iter=300, random_state=42)
labels_final = ap_final.fit_predict(X)
n_clusters_final = len(ap_final.cluster_centers_indices_)

print(f'Retreino com preference={best_pref}')
print(f'Clusters encontrados: {n_clusters_final}')

Retreino com preference=-50
Clusters encontrados: 7


## Passo 7 — Performance no teste

In [22]:
if 1 < n_clusters_final < len(X):
    sil_final = mt.silhouette_score(X, labels_final)
    print(f'--- Performance Final ---')
    print(f'Clusters         : {n_clusters_final}')
    print(f'Silhouette Score : {sil_final:.4f}')
else:
    sil_final = float('nan')
    print(f'Clusters: {n_clusters_final} — Silhouette Score não aplicável')

--- Performance Final ---
Clusters         : 7
Silhouette Score : 0.2023


## Passo 8 — Quadro Comparativo — Diagnóstico Completo

In [23]:
sil_def_str   = f'{sil_def:.4f}'   if sil_def == sil_def   else 'N/A'
sil_final_str = f'{sil_final:.4f}' if sil_final == sil_final else 'N/A'

data = {
    'Configuração':     ['Default', f'Tunado (preference={best_pref})'],
    'N° de Clusters':   [n_clusters_def,   n_clusters_final],
    'Silhouette Score': [sil_def_str, sil_final_str],
}
df_comp = pd.DataFrame(data)
print('--- Quadro Comparativo — Diagnóstico Completo ---')
print(df_comp.to_string(index=False))

--- Quadro Comparativo — Diagnóstico Completo ---
           Configuração  N° de Clusters Silhouette Score
                Default              17           0.1694
Tunado (preference=-50)               7           0.2023
